# 1장 실습 — 차량 80대는 적정한가

0장에서 하남시 택시 80대의 평균 대기시간이 4.1분이라는 결과를 봤습니다.
이번에는 코드를 구성하는 클래스와 객체부터 확인한 뒤, 평균 뒤에 가려진 승객과 차량을 찾습니다.

교재 1.3절에 대응합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, todo
from smartmob import Dtumos
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 클래스에서 객체를 만듭니다 (교재 0.4)

0장에서 `dt = Dtumos()` 를 실행했습니다. 이때 `Dtumos()` 가 무엇을 만들었는지 클래스 이름과 연결 상태로 확인합니다.
`type(dt).__name__` 은 객체가 어느 클래스에서 만들어졌는지 그 이름을 돌려줍니다.
`dt.mode` 를 처음 읽을 때 서버에 붙을 수 있는지 확인하고, 못 붙으면 `fixture` 로 정합니다.

In [ ]:
dt = Dtumos()
print("dt의 클래스:", type(dt).__name__)
print("연결 상태:", dt.health())
print("사용 중인 모드:", dt.mode)      # "fixture" 면 녹화본, "live" 면 실서버

첫 줄에 `Dtumos` 가 출력됩니다. 클래스(class)는 같은 종류의 객체가 가질 데이터와 동작을 정의합니다.
`Dtumos()` 로 만든 `dt` 는 객체(object)이며, 인스턴스(instance)라고도 부릅니다.

- `Dtumos` 는 클래스입니다
- `dt = Dtumos()` 는 객체를 만들어 `dt` 에 담습니다
- `dt.health()` 는 객체의 동작인 메서드(method)를 호출합니다
- `dt.mode` 는 객체가 가진 속성(attribute)을 읽습니다

메서드에는 `health()` 처럼 괄호가 붙고, 속성인 `mode` 에는 괄호가 없습니다.

## 2. 조건을 정하고 시뮬레이션을 실행합니다 (교재 1.3)

시뮬레이션 조건을 딕셔너리 `REFERENCE` 에 모읍니다.
`**REFERENCE` 는 딕셔너리의 키를 인자 이름으로, 값을 인자 값으로 풀어 넣는 문법입니다.
`run_simulation(city="hanam", mode="taxi", ...)` 이라고 길게 쓴 것과 같습니다.

실행 시간은 모드에 따라 다릅니다. 아래 셀은 먼저 `dt.mode` 를 찍습니다.

- `fixture`: 녹화본을 읽으므로 바로 끝납니다
- `live`: 실서버가 여섯 시간치 배차를 실제로 계산합니다. 몇 분 걸릴 수 있으니 기다립니다

인자는 녹화본과 같게 두었습니다. `fixture` 모드에서 값을 바꾸면 `FixtureMissing` 오류가 납니다.

In [ ]:
REFERENCE = dict(
    city="hanam",
    mode="taxi",
    fleet_size=80,
    num_passengers=1000,
    time_start=1080,                  # 18:00
    time_end=1440,                    # 24:00
    dispatch_mode="optimization",     # 배차 규칙. 여러 호출과 빈 차를 한꺼번에 짝지어 배정합니다 (10장)
    matrix_mode="street_distance",    # 차량과 승객 사이 거리를 직선이 아니라 도로망 경로로 잽니다 (3장)
    vehicle_capacity=1,               # 차량 한 대에 승객 한 명. 합승이 없습니다
    random_seed=42,                   # 난수 씨앗. 같은 값이면 같은 결과가 나옵니다
)

print("모드:", dt.mode)
sim = dt.run_simulation(**REFERENCE)
print("sim의 클래스:", type(sim).__name__)
sim.summary()

`SimulationResult` 가 출력됩니다. `run_simulation()` 은 실행 결과를 이 클래스의 객체로 돌려줍니다.
`summary()` 에는 호출 990건의 서비스율 1.0, 평균 대기시간 4.1분, 차량 가동률 0.27이 들어 있습니다.

`sim` 객체가 가진 것을 정리하면 이렇습니다.

| 이름 | 무엇인가 | 한 줄의 단위 |
|---|---|---|
| `sim.config` | 실행 조건 딕셔너리 | |
| `sim.passengers` | 승객별 호출·탑승 시각과 대기시간 | 승객 |
| `sim.record` | 0장에서 본 표. 컬럼이 `_cnt` 로 끝남 | 분 |
| `sim.result` | 차량 상태를 더 잘게 나눈 표. 컬럼이 `_num` 으로 끝남 | 분 |
| `sim.summary()` | 서비스율·평균 대기시간·가동률을 계산하는 메서드 | |

같은 `dt` 객체에 다른 조건을 넘기면 새 결과 객체가 나옵니다.
실서버가 있으면 차량 수만 바꿔 `sim` 과 `sim_40` 을 비교합니다.
녹화본에는 80대 결과만 있으므로 아래 코드는 읽기만 합니다. 서버를 띄우는 방법은 부록 B에 있습니다.

```python
alternative = {**REFERENCE, "fleet_size": 40}   # REFERENCE 를 복사하고 fleet_size 만 바꿉니다
sim_40 = dt.run_simulation(**alternative)
```

## 3. 승객별 대기시간을 읽습니다 (교재 1.3)

`sim.passengers` 는 승객마다 한 줄입니다. `status` 가 1이면 배차에 성공한 승객이고,
`wait_min` 은 호출부터 탑승까지 걸린 시간(분)입니다.
`request_time` 과 `pickup_time` 은 0장과 같은 자정 기준의 분 단위 시각입니다.

In [ ]:
pax = sim.passengers
print("승객 수:", len(pax))
pax[["passenger_id", "status", "request_time", "pickup_time", "wait_min"]].head()

승객이 990명입니다. 앞의 다섯 명은 18:01에 호출해 같은 분에 탔습니다.
시작 직후에는 빈 차가 80대라 바로 배차됩니다.

교재는 중앙값, 90% 분위, 95% 분위, 최댓값을 봅니다. 여기서는 평균을 더해 다섯 값을 나란히 놓습니다.
`quantile(0.95)` 는 승객을 대기시간 순으로 세웠을 때 95% 지점의 값입니다.
스무 명 중 한 명은 이보다 오래 기다렸다는 뜻입니다.

### 평균, 중앙값, 꼬리를 비교합니다

In [ ]:
waits = pax.loc[pax["status"] == 1, "wait_min"]   # 배차된 승객의 대기시간만 고릅니다

banner("대기시간 분포 (분)")
for label, value in [
    ("평균", waits.mean()),
    ("중앙값", waits.median()),
    ("90% 분위", waits.quantile(0.90)),
    ("95% 분위", waits.quantile(0.95)),
    ("최댓값", waits.max()),
]:
    print(f"{label:8s} {value:6.1f}")

절반은 3분 안에 탔지만, 스무 명 중 한 명은 11분 넘게 기다렸고 가장 오래 기다린 사람은 41분입니다.
평균 4.1분이라는 숫자 하나로는 이 사람이 보이지 않습니다.

### 히스토그램으로 분포를 확인합니다

`hist` 는 대기시간을 40개 구간으로 나눠 구간마다 승객 수를 셉니다.
세로 점선 두 개가 평균과 90% 분위입니다.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(waits, bins=40, color="#4C6EF5", edgecolor="white")
ax.axvline(waits.mean(), color="crimson", linestyle="--", label=f"평균 {waits.mean():.1f}분")       # 세로선
ax.axvline(waits.quantile(0.90), color="black", linestyle=":", label=f"90% 분위 {waits.quantile(0.90):.1f}분")
ax.set_xlabel("대기시간 (분)")
ax.set_ylabel("승객 수")
ax.legend()
ax.set_title("하남 택시 80대, 저녁 6시~자정")
plt.tight_layout();

왼쪽에 몰려 있고 오른쪽으로 길게 꼬리가 뻗습니다.
이런 모양에서는 평균만으로 모든 승객의 경험을 설명할 수 없습니다.

## 4. 오래 기다린 승객과 빈 차량을 찾습니다 (교재 1.3)

꼬리에 있는 승객이 특정 시간대에 몰려 있는지 봅니다.
`nlargest(10, "wait_min")` 은 `wait_min` 이 큰 순서로 10행을 고릅니다.
`map(minutes_to_hhmm)` 은 컬럼의 값 하나하나에 같은 함수를 적용합니다.

In [ ]:
from smartmob.data import minutes_to_hhmm

worst = pax.loc[pax["status"] == 1].nlargest(10, "wait_min")
worst = worst.assign(호출시각=worst["request_time"].map(minutes_to_hhmm))   # assign 은 새 컬럼을 붙인 표를 돌려줍니다
worst[["passenger_id", "호출시각", "wait_min"]]

상위 10명의 호출은 18:38~19:39와 22:13~23:17에 나뉘어 있습니다. 특정 한 시각의 문제로 좁힐 수 없습니다.

차량 쪽을 봅니다. 0장의 `sim.record` 는 차량을 빈 차와 운행 중인 차 둘로만 나눴습니다.
`sim.result` 는 운행 중인 차를 승객을 태운 차(`occupied`)와 태우러 가는 차(`dispatched`)로 다시 나눕니다.
`mean()` 은 분마다 기록된 대수를 여섯 시간에 걸쳐 평균 낸 값입니다.

In [ ]:
vehicle_state = sim.result

banner("시간대별 평균 차량 수")
print(f"승객을 태운 차량  {vehicle_state['occupied_vehicle_num'].mean():5.1f}대")
print(f"승객에게 가는 차량 {vehicle_state['dispatched_vehicle_num'].mean():5.1f}대")
print(f"빈 차량          {vehicle_state['empty_vehicle_num'].mean():5.1f}대")

승객을 태운 차량은 평균 10대, 태우러 가는 차량은 4대, 빈 차량은 31대입니다.
셋을 더해도 80대가 안 되는 것은 0장에서 본 대로 근무 시간이 끝난 차량이 빠지기 때문입니다.
오래 기다린 승객이 있는데도 빈 차량이 평균 31대이므로, 차량의 위치와 배차 규칙도 살펴봐야 합니다.

## 5. 빈칸

대기시간이 10분을 넘은 승객이 몇 명인지, 전체 배차 승객 중 몇 퍼센트인지 구합니다.
그리고 그 승객들의 평균 대기시간도 구합니다.

`waits > 10` 은 승객마다 참·거짓을 돌려줍니다. 참인 것의 개수는 `.sum()`, 비율은 `.mean()` 으로 셉니다.
`waits[waits > 10]` 처럼 대괄호 안에 조건을 넣으면 참인 승객만 남습니다.

In [ ]:
long_wait = None        # 10분을 넘게 기다린 승객 수 (명)
long_wait_share = None  # 전체 배차 승객 중 비율 (0~1)
long_wait_mean = None   # 그 승객들의 평균 대기시간 (분)

banner("빈칸 확인")
todo("10분 초과 승객", long_wait)
todo("그 비율", long_wait_share, fmt=lambda v: f"{v:.1%}")
todo("그들의 평균 대기", long_wait_mean, fmt=lambda v: f"{v:.1f}분")

## 6. 결과를 운영 판단으로 바꿉니다

80대 결과만으로 적정 차량 수를 확정할 수는 없습니다.
40대 결과와 비교하기 전에 어떤 지표를 볼지 아래 세 줄로 정리합니다. 답은 하나가 아닙니다.

1. 이 서비스를 "평균 4분"이라고 보고하면 무엇이 빠지는가
2. 대신 어떤 숫자를 함께 보고할 것인가. 그 이유는 무엇인가
3. 대기시간을 줄이려고 차량을 늘리면 무엇이 나빠지는가 (0장의 `utilization` 을 떠올립니다)

## 정리

- 클래스는 객체를 만드는 설계도이고, 객체의 동작은 메서드로 호출합니다
- `Dtumos` 객체에 실행 조건을 넘기면 `SimulationResult` 객체를 돌려받습니다
- 평균 4.1분 뒤에 95% 분위 11분, 최댓값 41분이 있습니다. 분포와 차량 상태를 함께 읽습니다
- `sim.passengers` 는 승객 단위, `sim.record` 와 `sim.result` 는 분 단위입니다
- 2장 실습에서는 이 시뮬레이션이 쓰는 도로망 데이터를 직접 엽니다